## Importing relevant libraries

In [15]:
import os
import requests
import pdfplumber
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select


__________________________

## Solar Power Generation Trends (Monthly) NITI 2015 - 2026

This script retrieves monthly solar electricity generation data from the NITI Aayog ICeD API and stores the raw dataset locally as a CSV file. Existing files are detected and skipped to avoid unnecessary API requests.


In [5]:
url = 'https://icedapi.niti.gov.in/v1/gen-metatable-data'

headers = {
    "accept": "application/json, text/plain, */*",
    "accept-language": "en-GB,en;q=0.9",
    "content-type": "application/json",
    "origin": "https://iced.niti.gov.in",
    "priority": "u=1, i",
    "referer": "https://iced.niti.gov.in/",
    "user-agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/153.0.0.0 Safari/537.36"
}

payload = {
    "source": "solar",
    "rangeType": "monthly"
}

# Path to store the raw csv
raw_path = "../data/raw/niti_aayog/solar-gen-trends-15to26.csv"


# Only trigger API call if local file doesn't exist
if not os.path.exists(raw_path):
    print(f'Fetching generation dataset from NITI')
    response = requests.post(
        url=url, 
        headers=headers, 
        json=payload
    )

    print(f'Status code: {response.status_code}')

    gen_data = response.json()

    #Checking if the fetched data is a list
    if isinstance(gen_data, list):
        gen_df = pd.DataFrame(gen_data)

    gen_df.to_csv('../data/raw/niti_aayog/solar-gen-trends-15to26.csv', index=False)

else: 
    print(f'Dataset already exists at {raw_path}. Skipping API request...')


gen_df

Dataset already exists at ../data/raw/niti_aayog/solar-gen-trends-15to26.csv. Skipping API request...


,source,state,generation,year,month
0,solar,Andaman and Nicobar Islands,1.17,2015-16,April
1,solar,Andaman and Nicobar Islands,0.83,2015-16,August
2,solar,Andaman and Nicobar Islands,0.84,2015-16,December
3,solar,Andaman and Nicobar Islands,1.37,2015-16,February
4,solar,Andaman and Nicobar Islands,1.35,2015-16,January
...,...,...,...,...,...
5251,solar,West Bengal,0.00,2026-27,March
5252,solar,West Bengal,47.38,2026-27,May
5253,solar,West Bengal,0.00,2026-27,November
5254,solar,West Bengal,0.00,2026-27,October


__________________________

## Solar Power Installed Capacity Trends (Yearly) NITI 2015 - 2026

This script retrieves installed power generation capacity data from the NITI Aayog ICeD API, filters the dataset to solar energy records, and saves the results as a local CSV file. Existing datasets are skipped to prevent redundant downloads.


In [6]:
url = "https://icedapi.niti.gov.in/v1/capacity-metatable-data?switchMetaData=false"

headers = {
    "accept" : "application/json, text/plain, */*",
    "accept-language": "en-GB, en; q=0.9",
    "dnt": "1", 
    "origin": "https://iced.niti.gov.in", 
    "referer": "https://iced.niti.gov.in/", 
    "user-agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/153.0.0.0 Safari/537.36"
}

raw_path = "../data/raw/niti_aayog/solar-installed-capacity-trends-15to26.csv"

# Only trigger API call if local file doesn't exist
if not os.path.exists(raw_path):
    print(f'Fetching capacity dataset from NITI')

    response = requests.get(url=url, headers=headers)

    print(f'Status code: {response.status_code}')

    capacity_data = response.json()
     #Checking if the fetched data is a list
    if isinstance(capacity_data, list):
        capcity_df = pd.DataFrame(capacity_data)

    capcity_df = capcity_df[capcity_df['source'].str.strip() =='solar']
    capcity_df.to_csv('../data/raw/niti_aayog/solar-installed-capacity-trends-15to26.csv', index=False)

else: 
    print(f'Dataset already exists at {raw_path}. Skipping API request...')

    
capcity_df

Fetching capacity dataset from NITI
Status code: 200


,state,fyear,source,capacity
1,Andaman and Nicobar Islands,2015-16,solar,5.34
4,Andaman and Nicobar Islands,2016-17,solar,6.80
7,Andaman and Nicobar Islands,2017-18,solar,6.80
10,Andaman and Nicobar Islands,2018-19,solar,11.97
13,Andaman and Nicobar Islands,2019-20,solar,12.43
...,...,...,...,...
2015,West Bengal,2022-23,solar,179.98
2021,West Bengal,2023-24,solar,194.07
2027,West Bengal,2024-25,solar,320.62
2032,West Bengal,2025-26,solar,320.62


## Energy Demand Trends CEA

### CEA Monthly Peak Demand Report Scraper

This script automates the download of CEA *Power Supply Position – Peak* reports from the Monthly Reports Archive. Using Selenium, it navigates the archive interface, selects the required year and month, locates the corresponding PDF report, and downloads it using Requests. Files are saved with a standardized naming convention (`PSP_Peak_YYYY-MM.pdf`) and existing downloads are skipped.


In [6]:
url = "https://cea.nic.in/monthly-reports-archive/?lang=en"
years = list(range(2015, 2027))
months = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]

#Chrome configuration objects
options = webdriver.ChromeOptions()
options.add_argument('--headless=new')
#Python interface for Selenium's browser
driver = webdriver.Chrome(options=options)


for year in years:
    for i, month in enumerate(months, start=1):
        
        #Checking if file already exists
        filename = f'../data/raw/cea/Demand/PSP_Peak_{year}-{i:02d}.pdf'
        if os.path.isfile(filename):
            print(f'Skipping {month} {year} as file already exits...')
            continue
        
        print(f'Checking {month} {year}...')

        #Getting the site from the browser
        driver.get(url)

        # finding dropdowns
        selects = driver.find_elements(By.TAG_NAME, 'select')

        year_dd, month_dd, type_dd = None, None, None

        for s in selects:
            options_text = [o.text.strip() for o in Select(s).options]

            if str(year) in options_text and '2015' in options_text:
                year_dd = Select(s)
            elif month in options_text and 'January' in options_text:
                month_dd = Select(s)
            elif 'Power Supply Reports' in options_text:
                type_dd = Select(s)
        
        if not (year_dd and month_dd and type_dd):
            print(f'could not find one of the dropdowns')
            continue

        year_dd.select_by_visible_text(str(year))
        month_dd.select_by_visible_text(month)
        type_dd.select_by_visible_text("Power Supply Reports")


        try:
            #finding Show archive reports button and clicking it
            driver.find_element(
                By.XPATH,
                "//*[contains(translate(text(),"
                "'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),"
                "'show archive reports')]"
            ).click()

        except Exception:
            print("  Could not find the 'Show Archive Reports' button, skipping.")
            continue

        # letting the page load
        time.sleep(3)

        pdf_url = None

        #Selecting rows
        rows = driver.find_elements(By.XPATH, "//*[contains(text(), 'Power Supply Position - Peak')]")

        for row in rows:
            #finding tr tag
            parent = row.find_element(By.XPATH, "./ancestor::tr[1]") if row.tag_name != "tr" else row
            #finding links
            links = parent.find_elements(By.XPATH, ".//a[@href]")
            #getting the link to the pdf
            for link in links:
                href = link.get_attribute("href") or ""
                if ".pdf" in href.lower():
                    pdf_url = href
                    break
            
            if pdf_url:
                break

        # if reports not found
        if not pdf_url:
            print('No peak report found for this month.')
            continue

        #Getting the required pdf
        response = requests.get(pdf_url, timeout=60)
        response.raise_for_status()
        #Writing into file
        with open(filename, "wb") as f:
            f.write(response.content)
            print(f'Saved {filename}')
        
driver.quit()
print("Done")



Skipping January 2026 as file already exits...
Skipping February 2026 as file already exits...
Skipping March 2026 as file already exits...
Skipping April 2026 as file already exits...
Skipping May 2026 as file already exits...
Skipping June 2026 as file already exits...
Skipping July 2026 as file already exits...
Skipping August 2026 as file already exits...
Checking September 2026...
No peak report found for this month.
Checking October 2026...
No peak report found for this month.
Checking November 2026...
No peak report found for this month.
Checking December 2026...
No peak report found for this month.
Done


### CEA PDF Extraction pipeline
This script extracts state-level peak demand data from the downloaded CEA PDF reports using `pdfplumber`. It cleans and standardizes state names, removes aggregate summary rows, handles merged PDF table cells, and converts the extracted information into a structured CSV dataset containing monthly peak demand and peak demand met values for each state.


In [ ]:
# 1. Directory containing downloaded PDFs
PDF_DIR = "../data/raw/cea/Demand"
OUTPUT_FILE = "../data/raw/cea/cea_csv/cea_monthly_peak_demand.csv"




# 2. Canonical state mapping to clean Hindi/English text and aliases
STATE_LOOKUP = {
    "chandigarh": "Chandigarh",
    "delhi": "Delhi",
    "haryana": "Haryana",
    "himachal": "Himachal Pradesh",
    "jammu": "Jammu and Kashmir",
    "j&k": "Jammu and Kashmir",
    "punjab": "Punjab",
    "rajasthan": "Rajasthan",
    "uttar pradesh": "Uttar Pradesh",
    "uttarakhand": "Uttarakhand",
    "chhattisgarh": "Chhattisgarh",
    "gujarat": "Gujarat",
    "madhya pradesh": "Madhya Pradesh",
    "maharashtra": "Maharashtra",
    "dadra": "Dadra and Nagar Haveli and Daman and Diu",
    "daman": "Dadra and Nagar Haveli and Daman and Diu",
    "goa": "Goa",
    "andhra": "Andhra Pradesh",
    "telangana": "Telangana",
    "karnataka": "Karnataka",
    "kerala": "Kerala",
    "tamil nadu": "Tamil Nadu",
    "puducherry": "Puducherry",
    "lakshadweep": "Lakshadweep",
    "bihar": "Bihar",
    "dvc": "DVC",
    "jharkhand": "Jharkhand",
    "odisha": "Odisha",
    "west bengal": "West Bengal",
    "sikkim": "Sikkim",
    "andaman": "Andaman and Nicobar",
    "arunachal": "Arunachal Pradesh",
    "assam": "Assam",
    "manipur": "Manipur",
    "meghalaya": "Meghalaya",
    "mizoram": "Mizoram",
    "nagaland": "Nagaland",
    "tripura": "Tripura",
}

# Words indicating aggregate summary rows to exclude
EXCLUDE_TERMS = [
    "region",
    "all india",
    "total",
    "system",
    "others",
    "eastern",
    "western",
    "northern",
    "southern",
]


def match_state(text):
    text_lower = text.lower()
    if any(term in text_lower for term in EXCLUDE_TERMS):
        return None
    for key, canonical_name in STATE_LOOKUP.items():
        if key in text_lower:
            return canonical_name
    return None


def clean_number(val):
    if not val:
        return None
    # Remove commas, whitespace, and non-numeric characters (except decimal)
    clean_val = re.sub(r"[^\d.]", "", str(val).strip())
    try:
        return float(clean_val)
    except ValueError:
        return None




if os.path.isfile(OUTPUT_FILE):
    print(f"{OUTPUT_FILE} already exists. Exiting...")
else:
    all_records = []
    pdf_files = sorted([f for f in os.listdir(PDF_DIR) if f.endswith(".pdf")])
    print(f"Found {len(pdf_files)} PDF files to process.")

    for filename in pdf_files:
        # Extract Year and Month from filename pattern 'PSP_Peak_YYYY-MM.pdf'
        date_match = re.search(r"(\d{4})-(\d{2})", filename)
        if not date_match:
            continue
        year = int(date_match.group(1))
        month = int(date_match.group(2))

        pdf_path = os.path.join(PDF_DIR, filename)

        with pdfplumber.open(pdf_path) as pdf:
            # Power supply position tables are always on page 1
            page = pdf.pages[0]
            tables = page.extract_tables()

            for table in tables:
                for row in table:
                    if not row or len(row) < 3:
                        continue

                    raw_state_cell = str(row[0]) if row[0] else ""
                    raw_demand_cell = str(row[1]) if row[1] else ""
                    raw_met_cell = str(row[2]) if row[2] else ""

                    # Handle multi-line cells where rows were merged due to missing borders
                    state_lines = [
                        s.strip() for s in raw_state_cell.split("\n") if s.strip()
                    ]
                    demand_lines = [
                        d.strip() for d in raw_demand_cell.split("\n") if d.strip()
                    ]
                    met_lines = [
                        m.strip() for m in raw_met_cell.split("\n") if m.strip()
                    ]

                    # If single row
                    if len(state_lines) <= 1:
                        state_name = match_state(raw_state_cell)
                        peak_demand = clean_number(raw_demand_cell)
                        peak_met = clean_number(raw_met_cell)

                        if state_name and peak_demand is not None:
                            all_records.append({
                                "state": state_name,
                                "year": year,
                                "month": month,
                                "peak_demand_mw": peak_demand,
                                "peak_met_mw": peak_met,
                                "source_file": filename,
                            })

                    # If multiple lines merged in the same row
                    else:
                        for idx, line in enumerate(state_lines):
                            state_name = match_state(line)
                            if state_name and idx < len(demand_lines):
                                peak_demand = clean_number(demand_lines[idx])
                                peak_met = (
                                    clean_number(met_lines[idx])
                                    if idx < len(met_lines)
                                    else None
                                )
                                if peak_demand is not None:
                                    all_records.append({
                                        "state": state_name,
                                        "year": year,
                                        "month": month,
                                        "peak_demand_mw": peak_demand,
                                        "peak_met_mw": peak_met,
                                        "source_file": filename,
                                    })

    # 3. Create DataFrame and deduplicate
    df = pd.DataFrame(all_records)
    df = df.drop_duplicates(subset=["state", "year", "month"])
    df = df.sort_values(by=["state", "year", "month"]).reset_index(drop=True)

    print("\n--- Extraction Summary ---")
    print(f"Total observations extracted: {len(df)}")
    print(f"Unique states captured: {df['state'].nunique()}")
    print(f"Date range: {df['year'].min()}-{df['month'].min():02d} to {df['year'].max()}-{df['month'].max():02d}")
    print("\nSample records:")
    print(df.head(10))

    # 4. Save to processed raw location
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"\nSaved clean demand data to {OUTPUT_FILE}")



../data/raw/cea/cea_csv/cea_monthly_peak_demand.csv already exists. Exiting.


### checking the df

In [10]:
demand_df = pd.read_csv("../data/raw/cea/cea_csv/cea_monthly_peak_demand.csv")

demand_df


,state,year,month,peak_demand_mw,peak_met_mw,source_file
0,Andhra Pradesh,2015,1,6397.0,6392.0,PSP_Peak_2015-01.pdf
1,Andhra Pradesh,2015,2,6530.0,6518.0,PSP_Peak_2015-02.pdf
2,Andhra Pradesh,2015,3,6790.0,6784.0,PSP_Peak_2015-03.pdf
3,Andhra Pradesh,2015,4,6794.0,6789.0,PSP_Peak_2015-04.pdf
4,Andhra Pradesh,2015,5,6737.0,6732.0,PSP_Peak_2015-05.pdf
...,...,...,...,...,...,...
4879,West Bengal,2026,4,13023.0,12946.0,PSP_Peak_2026-04.pdf
4880,West Bengal,2026,5,13340.0,13340.0,PSP_Peak_2026-05.pdf
4881,West Bengal,2026,6,13570.0,13570.0,PSP_Peak_2026-06.pdf
4882,West Bengal,2026,7,12324.0,12324.0,PSP_Peak_2026-07.pdf


## NASA Power for weather data

In [16]:
# Coordinates of centroids for Indian States and UTs
STATE_COORDINATES = {
    # States
    "Andhra Pradesh": (15.9129, 79.7400),
    "Arunachal Pradesh": (28.2180, 94.7278),
    "Assam": (26.2006, 92.9376),
    "Bihar": (25.0961, 85.3131),
    "Chhattisgarh": (21.2787, 81.8661),
    "Goa": (15.2993, 74.1240),
    "Gujarat": (22.2587, 71.1924),
    "Haryana": (29.0588, 76.0856),
    "Himachal Pradesh": (31.1048, 77.1734),
    "Jharkhand": (23.6102, 85.2799),
    "Karnataka": (15.3173, 75.7139),
    "Kerala": (10.8505, 76.2711),
    "Madhya Pradesh": (22.9734, 78.6569),
    "Maharashtra": (19.7515, 75.7139),
    "Manipur": (24.6637, 93.9063),
    "Meghalaya": (25.4670, 91.3662),
    "Mizoram": (23.1645, 92.9376),
    "Nagaland": (26.1584, 94.5624),
    "Odisha": (20.9517, 85.0985),
    "Punjab": (31.1471, 75.3412),
    "Rajasthan": (27.0238, 74.2179),
    "Sikkim": (27.5330, 88.5122),
    "Tamil Nadu": (11.1271, 78.6569),
    "Telangana": (18.1124, 79.0193),
    "Tripura": (23.9408, 91.9882),
    "Uttar Pradesh": (26.8467, 80.9462),
    "Uttarakhand": (30.0668, 79.0193),
    "West Bengal": (22.9868, 87.8550),

    # Union Territories
    "Andaman and Nicobar Islands": (11.7401, 92.6586),
    "Chandigarh": (30.7333, 76.7794),
    "Dadra and Nagar Haveli and Daman and Diu": (20.1809, 73.0169),
    "Delhi": (28.7041, 77.1025),
    "Jammu and Kashmir": (33.7782, 76.5762),
    "Ladakh": (34.1526, 77.5771),
    "Lakshadweep": (10.5667, 72.6417),
    "Puducherry": (11.9416, 79.8083),
}


BASE_URL = "https://power.larc.nasa.gov/api/temporal/monthly/point"
START_YEAR = 2015
END_YEAR = 2025

weather_records = []

print('Starting NASA Power API extraction...')

for state, (lat, long) in STATE_COORDINATES.items():
    print(f"Fetching weather data for: {state} ({lat, long})")

    params = {
        "parameters": "ALLSKY_SFC_SW_DWN,T2M",
        "community": "RE",
        "longitude": long,
        "latitude": lat,
        "start": START_YEAR,
        "end": END_YEAR,
        "format": "JSON"
    }

    try:
        response = requests.get(BASE_URL, params=params, timeout = 30)
        if response.status_code == 200:
            data = response.json()
            irradiance_dict = data["properties"]["parameter"]["ALLSKY_SFC_SW_DWN"]
            temp_dict = data["properties"]["parameter"]["T2M"]
            
    except Exception as e:
        print(f'Error fetching {state}: {e}')


print(irradiance_dict)
print(temp_dict)

Starting NASA Power API extraction...
Fetching weather data for: Andhra Pradesh ((15.9129, 79.74))
Fetching weather data for: Arunachal Pradesh ((28.218, 94.7278))
Fetching weather data for: Assam ((26.2006, 92.9376))
Fetching weather data for: Bihar ((25.0961, 85.3131))
Fetching weather data for: Chhattisgarh ((21.2787, 81.8661))
Fetching weather data for: Goa ((15.2993, 74.124))
Fetching weather data for: Gujarat ((22.2587, 71.1924))
Fetching weather data for: Haryana ((29.0588, 76.0856))
Fetching weather data for: Himachal Pradesh ((31.1048, 77.1734))
Fetching weather data for: Jharkhand ((23.6102, 85.2799))
Fetching weather data for: Karnataka ((15.3173, 75.7139))
Fetching weather data for: Kerala ((10.8505, 76.2711))
Fetching weather data for: Madhya Pradesh ((22.9734, 78.6569))
Fetching weather data for: Maharashtra ((19.7515, 75.7139))
Fetching weather data for: Manipur ((24.6637, 93.9063))
Fetching weather data for: Meghalaya ((25.467, 91.3662))
Fetching weather data for: Mizor